In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Transform: converting images to tensors and normalize
transform = transforms.Compose([
    transforms.ToTensor(),                               # Converts PIL image to tensor
    transforms.Normalize((0.1307,),(0.3081,))            # Normalize with MNIST mean and std where the values are previously are calculated.
])

# MNIST dataset 
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

(X_train, y_train) = train_dataset.data, train_dataset.targets
(X_test, y_test) = test_dataset.data, test_dataset.targets

X_train = X_train / 255.0
X_test = X_test / 255.0

encoder = OneHotEncoder(sparse_output=False)
y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_oh = encoder.transform(y_test.reshape(-1, 1))

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("Y Training data shape:", y_train.shape)
print("Y Testing data shape:", y_test.shape)


Training data shape: torch.Size([60000, 28, 28])
Testing data shape: torch.Size([10000, 28, 28])
Y Training data shape: torch.Size([60000])
Y Testing data shape: torch.Size([10000])


In [6]:
il, hl, ol, lr = 784, [128], 10, 0.01
batch_size = 64
epoch = 5

np.random.seed(42)
torch.manual_seed(42)

def mlp(il, hl, ol):
    layers = []
    prev = il
    for h in hl:
        layers.append(nn.Linear(prev,h))
        layers.append(nn.ReLU())
        prev = h
    layers.append(nn.Linear(prev, ol))
    return nn.Sequential(*layers)

model = mlp(il, hl, ol)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Convert to torch tensors
X_train = X_train.float()
y_train = y_train.long()
X_test = X_test.float()
y_test = y_test.long()


##### Manual Mini-Batch

In [8]:

for epoch in range(5):
    model.train()

    # Shuffle training data
    indices = torch.randperm(X_train.size(0))
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]

    # Split into mini-batches
    X_batches = torch.split(X_train_shuffled, batch_size)
    y_batches = torch.split(y_train_shuffled, batch_size)

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in zip(X_batches, y_batches):
        optimizer.zero_grad()

        outputs = model(batch_X.view(batch_X.size(0), -1))
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2%}")


Epoch 1, Loss: 0.0869, Accuracy: 97.68%
Epoch 2, Loss: 0.0837, Accuracy: 97.79%
Epoch 3, Loss: 0.0818, Accuracy: 97.90%
Epoch 4, Loss: 0.0748, Accuracy: 98.12%
Epoch 5, Loss: 0.0726, Accuracy: 98.21%


#### using Data Loder

In [7]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

for epoch in range(epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X.view(batch_X.size(0), -1))
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs, -1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2%}")

Epoch 1, Loss: 0.2244, Accuracy: 93.21%
Epoch 2, Loss: 0.1292, Accuracy: 96.29%
Epoch 3, Loss: 0.1145, Accuracy: 96.75%
Epoch 4, Loss: 0.0985, Accuracy: 97.17%
Epoch 5, Loss: 0.0901, Accuracy: 97.52%


In [ ]:
# def base_fn(model_class = MLP_base(), lr = 0.01):
#     batch_size, lr = 64, lr
#     epoch = 5
#     model = model_class
#     criterion = nn.CrossEntropyLoss()
#     optimizer = optim.Adam(model.parameters(), lr=lr)

#     train_dataset = TensorDataset(X_train, y_train)
#     test_dataset = TensorDataset(X_test, y_test)

#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#     test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#     for epoch in range(epoch):
#         model.train()
#         running_loss = 0.0
#         correct = 0
#         total = 0

#         for batch_X, batch_y in train_loader:
#             optimizer.zero_grad()
#             outputs = model(batch_X.view(batch_X.size(0), -1))
#             loss = criterion(outputs, batch_y)
#             loss.backward()
#             optimizer.step()

#             running_loss += loss.item() * batch_X.size(0)
#             _, predicted = torch.max(outputs, -1)
#             correct += (predicted == batch_y).sum().item()
#             total += batch_y.size(0)

#         epoch_loss = running_loss / total
#         epoch_acc = correct / total

#         if (epoch + 1) % 1 == 0:
#             print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2%}")

# base_fn()